# PAPER-EXP-08A — 06D Gain 1.05 Final Configuration Lock (v2)

This notebook accepts the exact PAPER-EXP-06A and PAPER-EXP-06D Recovery materials whether Kaggle exposes the original ZIP archives or extracted dataset files. It verifies exact component SHA256 hashes and does not read frozen TEST or DERM12345 material.


In [1]:
#!/usr/bin/env python3
from __future__ import annotations

import hashlib
import io
import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

CODE_VERSION = "PAPEREXP08A_06D_GAIN_1P05_FINAL_CONFIGURATION_LOCK_2026-08-30_v2_KAGGLE_EXTRACTED_RECOVERY_SUPPORT"
LOCK_ID = "PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK_2026-08-30_v1"
GAIN = 1.05

CLASSES = ("MEL", "NV", "BCC", "AKIEC", "BKL")
DOMAINS = ("ISIC2018_HAM10000", "BCN20000", "Derm7pt", "PAD_UFES_20")
PRESENT = {
    "ISIC2018_HAM10000": ("MEL", "NV", "BCC", "AKIEC", "BKL"),
    "BCN20000": ("MEL", "NV", "BCC", "AKIEC", "BKL"),
    "Derm7pt": ("MEL", "NV", "BCC", "BKL"),
    "PAD_UFES_20": ("MEL", "NV", "BCC", "AKIEC", "BKL"),
}

EXPECTED_06A_RECOVERY_SHA = "f426562e03e694706052ca324434d301c229be8db3338eb209f3b93d680622db"
EXPECTED_06D_RECOVERY_SHA = "debe5c7068f421bae3afdd5412cc7b718a8128a2a3b1848f2445ed3c13bd809c"

CHECKPOINTS_06A = {
    "checkpoints/b0_seed_20260820/best.pth": "51eb6d967c6805ad622863c9b2d586954a119c2330a6608e9a73c95b808e6932",
    "checkpoints/b0_seed_20260821/best.pth": "a32e5e6679af64db64d7154d59dcf62d7f8ef230b3f029d0c610168ea49998c3",
    "checkpoints/b0_seed_20260822/best.pth": "610a79ffda7719598d6b26009dfe1c5534a4c7951179f9b018a5381e04bf7f6d",
    "checkpoints/b3_seed_20260820/best.pth": "a5d882e39255417a7171a5403b4298430d8ac0c3af7e31dd589538c73e950b19",
    "checkpoints/b3_seed_20260821/best.pth": "8827e01a59205723c5fefad9449740129072edb07bd4870c1361d904e512bbdb",
    "checkpoints/b3_seed_20260822/best.pth": "33b60778f13785ca7748d85a32421bb53a9044f23ed716aafb1c625238463e98",
}
CHECKPOINTS_06D = {
    "checkpoints/adapter_seed_20260829/selected.pth": "d96365b453a9306955f0025930629710d49e30a1eaf1950ca1784371afd6c667",
    "checkpoints/adapter_seed_20260830/selected.pth": "93ee73ffe2b6b4f65aa2ba61881b18646cf8a874324d3a60018e89c0ac77014d",
    "checkpoints/adapter_seed_20260831/selected.pth": "03b443c8e0b9bf9dd886397fbc45ae49ad0b18556919edb58f90a17ca0949e83",
}

BASE_REL = "validation_predictions/BASELINE_06A_fixed_50B0_50B3_tta7_validation.csv"
CAND_REL = "validation_predictions/PAPEREXP06D_soft_routed_residual_tta7_validation.csv"
BASE_SHA = "60ca81603aaf03486e776f80def3ca086ba118ea9a38838b7b071b1e06ec3a20"
CAND_SHA = "920bc6748af41943a6d8026ab569db4e6aaa8bc5f8cd5b3cffcde2ba6027e26a"
DECISION_REL = "PAPEREXP06D_FINAL_DECISION.json"
DECISION_SHA = "8b8722aa2aeff0c16d7671f7bdf2207e1f17087931aa50a71bf2d70544870c23"
PROTOCOL_REL = "paper_exp06d_protocol.json"
PROTOCOL_SHA = "07f167405093bb78379a352f0c742227c38b093f4b891fb23c1832a9d1b1777d"

EXPECTED_08A = {
    "equal_domain_accuracy": 0.8164224620059601,
    "equal_domain_macro_f1": 0.7732056009568344,
    "equal_domain_balanced_accuracy": 0.7675812016694961,
    "equal_domain_melanoma_recall": 0.692299414600551,
    "equal_domain_macro_auc": 0.9491443669860594,
}


def sha_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def candidate_roots():
    # Optional override is used only for local implementation testing. Kaggle needs no setting.
    override = os.environ.get("PAPEREXP08A_INPUT_ROOT", "").strip()
    if override:
        roots = [Path(override)]
    else:
        roots = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]
    return [r for r in roots if r.exists()]


def try_find_exact_archive(expected_sha: str, filename_tokens: tuple[str, ...]) -> Path | None:
    candidates: list[Path] = []
    for root in candidate_roots():
        for p in root.rglob("*.zip"):
            lower = p.name.lower()
            if all(token.lower() in lower for token in filename_tokens):
                candidates.append(p)
    for p in candidates:
        try:
            if sha_file(p) == expected_sha:
                return p
        except Exception:
            continue
    return None


def find_exact_extracted_file(rel: str, expected_sha: str) -> Path:
    # Kaggle commonly expands an uploaded ZIP when it becomes a Dataset. In that case
    # the original ZIP-container SHA cannot be recomputed, so we verify the exact
    # required file bytes by their predeclared SHA256 instead.
    basename = Path(rel).name
    checked = []
    exact = []
    for root in candidate_roots():
        for p in root.rglob(basename):
            if not p.is_file():
                continue
            try:
                observed = sha_file(p)
            except Exception:
                continue
            checked.append((str(p), observed))
            if observed == expected_sha:
                exact.append(p)
    if not exact:
        raise FileNotFoundError(
            f"Could not find exact extracted material {rel!r} / SHA256 {expected_sha}. "
            f"Checked={checked[:30]}"
        )
    # Multiple byte-identical copies are harmless; choose deterministically.
    return sorted(exact, key=lambda p: (len(str(p)), str(p)))[0]


def exact_member(zf: zipfile.ZipFile, rel: str, expected_sha: str) -> tuple[str, bytes]:
    matches = [n for n in zf.namelist() if n == rel or n.endswith("/" + rel)]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one member ending in {rel!r}; found {matches}")
    name = matches[0]
    data = zf.read(name)
    observed = sha_bytes(data)
    if observed != expected_sha:
        raise RuntimeError(f"SHA256 mismatch for {rel}: expected {expected_sha}, observed {observed}")
    return name, data


def softmax(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def macro_auc_present(y: np.ndarray, probs: np.ndarray, ids: list[int]) -> float:
    aucs = []
    for i in ids:
        target = (y == i).astype(np.int64)
        if target.min() == target.max():
            continue
        aucs.append(float(roc_auc_score(target, probs[:, i])))
    return float(np.mean(aucs)) if aucs else float("nan")


def domain_metrics(frame: pd.DataFrame, probs: np.ndarray) -> dict:
    y = frame["label"].to_numpy(dtype=int)
    ds = frame["dataset"].astype(str).to_numpy()
    pred = probs.argmax(axis=1)
    per = {}
    for d in DOMAINS:
        mask = ds == d
        yy = y[mask]
        yp = pred[mask]
        pp = probs[mask]
        ids = [CLASSES.index(c) for c in PRESENT[d]]
        recalls = [float(np.mean(yp[yy == i] == i)) for i in ids if np.any(yy == i)]
        mel_mask = yy == CLASSES.index("MEL")
        per[d] = {
            "n": int(mask.sum()),
            "accuracy": float(accuracy_score(yy, yp)),
            "macro_f1": float(f1_score(yy, yp, labels=ids, average="macro", zero_division=0)),
            "balanced_accuracy": float(np.mean(recalls)),
            "melanoma_recall": float(np.mean(yp[mel_mask] == CLASSES.index("MEL"))),
            "macro_auc": macro_auc_present(yy, pp, ids),
        }
    result = {"per_domain": per}
    for metric in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"):
        result[f"equal_domain_{metric}"] = float(np.mean([per[d][metric] for d in DOMAINS]))
    return result


def verify_expected_metrics(metrics: dict, tol: float = 2e-10) -> None:
    for key, expected in EXPECTED_08A.items():
        observed = float(metrics[key])
        if abs(observed - expected) > tol:
            raise RuntimeError(
                f"08A validation reproduction mismatch for {key}: expected {expected}, observed {observed}"
            )


def write_json(path: Path, obj: dict) -> None:
    path.write_text(json.dumps(obj, indent=2, sort_keys=False), encoding="utf-8")


def build_index(root: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.name != "paper_material_index.csv":
            rows.append({
                "relative_path": p.relative_to(root).as_posix(),
                "size_bytes": p.stat().st_size,
                "sha256": sha_file(p),
            })
    frame = pd.DataFrame(rows)
    frame.to_csv(root / "paper_material_index.csv", index=False)
    return frame


def zip_tree(root: Path, destination: Path, compression=zipfile.ZIP_DEFLATED) -> None:
    if destination.exists():
        destination.unlink()
    with zipfile.ZipFile(destination, "w", compression=compression, allowZip64=True) as zf:
        for p in sorted(root.rglob("*")):
            if p.is_file():
                zf.write(p, p.relative_to(root).as_posix())


def main():
    print("[CODE VERSION]", CODE_VERSION)
    print("[LOCK ID]", LOCK_ID)
    print("[STATUS] DEVELOPMENT VALIDATION REPRODUCTION + FINAL CONFIGURATION LOCK")
    print("[BOUNDARY] NO FROZEN TEST OR DERM12345 FILE IS READ BY THIS NOTEBOOK")

    recovery06a = try_find_exact_archive(
        EXPECTED_06A_RECOVERY_SHA,
        ("paperexp06a", "recovery"),
    )
    recovery06d = try_find_exact_archive(
        EXPECTED_06D_RECOVERY_SHA,
        ("paperexp06d", "recovery"),
    )
    mode06a = "ORIGINAL_ZIP" if recovery06a is not None else "KAGGLE_EXTRACTED_FILES"
    mode06d = "ORIGINAL_ZIP" if recovery06d is not None else "KAGGLE_EXTRACTED_FILES"
    print("[06A SOURCE MODE]", mode06a, recovery06a if recovery06a else "verified by exact component SHA256")
    print("[06D SOURCE MODE]", mode06d, recovery06d if recovery06d else "verified by exact component SHA256")

    output_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
    result_dir = output_root / "PAPEREXP08A_Final_Validation_Refined_Generalized_Model_Results"
    lock_dir = output_root / "PAPEREXP08A_Final_Locked_Generalized_Model_Recovery"
    shutil.rmtree(result_dir, ignore_errors=True)
    shutil.rmtree(lock_dir, ignore_errors=True)
    result_dir.mkdir(parents=True)
    lock_dir.mkdir(parents=True)

    component_audit = {
        "06A_source_mode": mode06a,
        "06D_source_mode": mode06d,
        "06A_expected_original_recovery_sha256": EXPECTED_06A_RECOVERY_SHA,
        "06D_expected_original_recovery_sha256": EXPECTED_06D_RECOVERY_SHA,
        "06A_original_zip_sha256_verified": bool(recovery06a is not None),
        "06D_original_zip_sha256_verified": bool(recovery06d is not None),
    }
    if recovery06a is not None:
        component_audit["06A_observed_recovery_sha256"] = sha_file(recovery06a)
    if recovery06d is not None:
        component_audit["06D_observed_recovery_sha256"] = sha_file(recovery06d)

    # Verify and extract exact six frozen 06A backbone checkpoints.
    extracted_06a = {}
    if recovery06a is not None:
        with zipfile.ZipFile(recovery06a) as z06a:
            for rel, expected_sha in CHECKPOINTS_06A.items():
                name, data = exact_member(z06a, rel, expected_sha)
                target = lock_dir / "checkpoints" / "06A" / rel.replace("checkpoints/", "")
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
                extracted_06a[rel] = {"source": str(recovery06a) + "::" + name, "sha256": sha_bytes(data), "size_bytes": len(data)}
    else:
        for rel, expected_sha in CHECKPOINTS_06A.items():
            source = find_exact_extracted_file(rel, expected_sha)
            data = source.read_bytes()
            target = lock_dir / "checkpoints" / "06A" / rel.replace("checkpoints/", "")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            extracted_06a[rel] = {"source": str(source), "sha256": sha_bytes(data), "size_bytes": len(data)}

    # Verify 06D metadata, validation probabilities and exact three selected adapter checkpoints.
    if recovery06d is not None:
        with zipfile.ZipFile(recovery06d) as z06d:
            _, decision_bytes = exact_member(z06d, DECISION_REL, DECISION_SHA)
            _, protocol_bytes = exact_member(z06d, PROTOCOL_REL, PROTOCOL_SHA)
            _, base_bytes = exact_member(z06d, BASE_REL, BASE_SHA)
            _, cand_bytes = exact_member(z06d, CAND_REL, CAND_SHA)
            extracted_06d = {}
            for rel, expected_sha in CHECKPOINTS_06D.items():
                name, data = exact_member(z06d, rel, expected_sha)
                target = lock_dir / "checkpoints" / "06D_adapters" / rel.replace("checkpoints/", "")
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
                extracted_06d[rel] = {"source": str(recovery06d) + "::" + name, "sha256": sha_bytes(data), "size_bytes": len(data)}
    else:
        decision_path = find_exact_extracted_file(DECISION_REL, DECISION_SHA)
        protocol_path = find_exact_extracted_file(PROTOCOL_REL, PROTOCOL_SHA)
        base_path = find_exact_extracted_file(BASE_REL, BASE_SHA)
        cand_path = find_exact_extracted_file(CAND_REL, CAND_SHA)
        decision_bytes = decision_path.read_bytes()
        protocol_bytes = protocol_path.read_bytes()
        base_bytes = base_path.read_bytes()
        cand_bytes = cand_path.read_bytes()
        extracted_06d = {}
        for rel, expected_sha in CHECKPOINTS_06D.items():
            source = find_exact_extracted_file(rel, expected_sha)
            data = source.read_bytes()
            target = lock_dir / "checkpoints" / "06D_adapters" / rel.replace("checkpoints/", "")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            extracted_06d[rel] = {"source": str(source), "sha256": sha_bytes(data), "size_bytes": len(data)}

    decision06d = json.loads(decision_bytes.decode("utf-8"))
    protocol06d = json.loads(protocol_bytes.decode("utf-8"))

    base = pd.read_csv(io.BytesIO(base_bytes))
    cand = pd.read_csv(io.BytesIO(cand_bytes))
    keys = ["dataset", "image_id_norm", "group_id", "class_name", "label"]
    if not base[keys].astype(str).equals(cand[keys].astype(str)):
        raise RuntimeError("06A/06D validation prediction alignment mismatch")

    prob_cols = [f"prob_{c}" for c in CLASSES]
    p0 = base[prob_cols].to_numpy(dtype=np.float64)
    p1 = cand[prob_cols].to_numpy(dtype=np.float64)
    if not np.isfinite(p0).all() or not np.isfinite(p1).all():
        raise RuntimeError("Non-finite source validation probabilities")
    p0 /= p0.sum(axis=1, keepdims=True)
    p1 /= p1.sum(axis=1, keepdims=True)

    # Reconstruct the already-learned 06D residual in centered log-probability space.
    residual = np.log(np.clip(p1, 1e-12, 1.0)) - np.log(np.clip(p0, 1e-12, 1.0))
    residual -= residual.mean(axis=1, keepdims=True)
    p_refined = softmax(np.log(np.clip(p0, 1e-12, 1.0)) + GAIN * residual)

    metrics06d = domain_metrics(cand, p1)
    metrics08a = domain_metrics(cand, p_refined)
    verify_expected_metrics(metrics08a)

    pred = cand[keys].copy()
    y_pred = p_refined.argmax(axis=1)
    pred["model_name"] = "PAPER-EXP-08A final validation-refined 06D configuration"
    pred["residual_gain"] = GAIN
    pred["predicted_label"] = y_pred
    pred["predicted_class"] = [CLASSES[i] for i in y_pred]
    pred["correct"] = (y_pred == pred["label"].to_numpy(dtype=int)).astype(int)
    for i, c in enumerate(CLASSES):
        pred[f"prob_{c}"] = p_refined[:, i]
    pred_path = result_dir / "PAPEREXP08A_validation_predictions.csv"
    pred.to_csv(pred_path, index=False)

    deltas = {
        k: float(metrics08a[k] - metrics06d[k])
        for k in (
            "equal_domain_accuracy",
            "equal_domain_macro_f1",
            "equal_domain_balanced_accuracy",
            "equal_domain_melanoma_recall",
            "equal_domain_macro_auc",
        )
    }

    gate_checks = {
        "equal_domain_macro_f1_non_decrease": deltas["equal_domain_macro_f1"] >= 0.0,
        "equal_domain_accuracy_non_decrease": deltas["equal_domain_accuracy"] >= 0.0,
        "equal_domain_balanced_accuracy_min_delta_-0.001": deltas["equal_domain_balanced_accuracy"] >= -0.001,
        "equal_domain_melanoma_recall_min_delta_-0.001": deltas["equal_domain_melanoma_recall"] >= -0.001,
        "ISIC_macro_f1_min_delta_-0.001": metrics08a["per_domain"]["ISIC2018_HAM10000"]["macro_f1"] - metrics06d["per_domain"]["ISIC2018_HAM10000"]["macro_f1"] >= -0.001,
        "BCN_macro_f1_non_decrease": metrics08a["per_domain"]["BCN20000"]["macro_f1"] >= metrics06d["per_domain"]["BCN20000"]["macro_f1"],
        "Derm7pt_macro_f1_min_delta_-0.001": metrics08a["per_domain"]["Derm7pt"]["macro_f1"] - metrics06d["per_domain"]["Derm7pt"]["macro_f1"] >= -0.001,
        "PAD_macro_f1_non_decrease": metrics08a["per_domain"]["PAD_UFES_20"]["macro_f1"] >= metrics06d["per_domain"]["PAD_UFES_20"]["macro_f1"],
        "PAD_accuracy_min_delta_-0.001": metrics08a["per_domain"]["PAD_UFES_20"]["accuracy"] - metrics06d["per_domain"]["PAD_UFES_20"]["accuracy"] >= -0.001,
    }
    if not all(gate_checks.values()):
        raise RuntimeError(f"08A validation refinement gate failed unexpectedly: {gate_checks}")

    validation_decision = {
        "experiment": "PAPER-EXP-08A",
        "code_version": CODE_VERSION,
        "status": "VALIDATION_REPRODUCED_AND_CONFIGURATION_LOCKED",
        "residual_gain": GAIN,
        "06D_validation": metrics06d,
        "08A_validation": metrics08a,
        "validation_deltas_vs_06D": deltas,
        "gate_checks": gate_checks,
        "gate_passed": True,
        "frozen_test_read": False,
        "DERM12345_read": False,
    }
    write_json(result_dir / "PAPEREXP08A_VALIDATION_DECISION.json", validation_decision)

    lock = {
        "lock_id": LOCK_ID,
        "experiment": "PAPER-EXP-08A",
        "configuration_name": "06D frozen-backbone soft-routed residual ensemble with global residual gain 1.05",
        "status": "FINAL_VALIDATION_SELECTED_GENERALIZED_CONFIGURATION",
        "locked_at_code_version": CODE_VERSION,
        "model_architecture": "IDENTICAL_TO_PAPER_EXP_06D",
        "training_change_from_06D": "NONE",
        "weights_change_from_06D": "NONE",
        "only_inference_change": {"global_residual_gain": GAIN},
        "inference_definition": {
            "class_order": list(CLASSES),
            "base_system": "PAPER-EXP-06A fixed six-checkpoint B0/B3 ensemble as used by PAPER-EXP-06D",
            "adapter_system": "mean residual from the three selected PAPER-EXP-06D epoch-3 soft-routed residual adapters",
            "formula": "P_final = softmax(log(P_06A) + 1.05 * R_06D), where R_06D is the centered log-probability residual reconstructed/implemented by the unchanged 06D adapter system",
            "residual_gain": GAIN,
            "tta": "unchanged from PAPER-EXP-06D (fixed TTA7)",
            "B0_B3_fusion": "unchanged from PAPER-EXP-06D (fixed 0.5/0.5 family fusion)",
            "adapter_seed_aggregation": "unchanged from PAPER-EXP-06D (mean of three selected adapter seeds)",
            "thresholds": "argmax; no threshold tuning",
            "calibration": "none",
        },
        "source_archives": {
            "PAPEREXP06A_recovery_sha256": EXPECTED_06A_RECOVERY_SHA,
            "PAPEREXP06D_recovery_sha256": EXPECTED_06D_RECOVERY_SHA,
        },
        "locked_06A_checkpoint_sha256": CHECKPOINTS_06A,
        "locked_06D_adapter_checkpoint_sha256": CHECKPOINTS_06D,
        "source_validation_material_sha256": {
            BASE_REL: BASE_SHA,
            CAND_REL: CAND_SHA,
            DECISION_REL: DECISION_SHA,
            PROTOCOL_REL: PROTOCOL_SHA,
        },
        "selected_common_adapter_epoch": int(decision06d.get("selected_common_epoch", 3)),
        "validation_evidence": metrics08a,
        "validation_gate_passed": True,
        "scientific_claim_boundary": {
            "validation_selected_after_prior_frozen_evaluation": True,
            "previously_opened_frozen_partitions_are_not_independent_evidence_for_this_08A_configuration": True,
            "do_not_tune_on_or_reselect_using_previously_opened_frozen_test_results": True,
            "new_untouched_cohort_required_for_a_new_independent_final_test_claim": True,
            "DERM12345_was_not_used_for_08A_selection": True,
        },
        "future_change_policy": "If this configuration is treated as the final generalized configuration, do not change gain, checkpoints, router, experts, TTA, fusion, thresholds, or calibration before a new untouched evaluation.",
    }
    write_json(lock_dir / "PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK.json", lock)
    write_json(lock_dir / "PAPEREXP08A_VALIDATION_DECISION.json", validation_decision)
    write_json(lock_dir / "PAPEREXP06D_FINAL_DECISION_SOURCE.json", decision06d)
    write_json(lock_dir / "paper_exp06d_protocol_source.json", protocol06d)

    readme = f"""# PAPER-EXP-08A Final Locked Generalized Configuration\n\nLock ID: `{LOCK_ID}`\n\nThis archive is self-contained for the learned weights: it includes the exact six PAPER-EXP-06A backbone checkpoints and the exact three selected PAPER-EXP-06D adapter checkpoints. The only PAPER-EXP-08A change is the fixed global residual gain `{GAIN}`.\n\n## Scientific boundary\n\nPAPER-EXP-08A was selected from development validation after the earlier frozen partitions had already been opened under PAPER-EXP-07A. Therefore those previously opened partitions cannot serve as a new independent final test for this refined configuration. Do not use them for tuning or re-selection. A new untouched cohort is required for a new independent final-test claim.\n\nDERM12345 was not read or used by this notebook.\n"""
    (lock_dir / "README.md").write_text(readme, encoding="utf-8")

    component_audit["06A_checkpoints"] = extracted_06a
    component_audit["06D_adapters"] = extracted_06d
    component_audit["source_material"] = {
        BASE_REL: BASE_SHA,
        CAND_REL: CAND_SHA,
        DECISION_REL: DECISION_SHA,
        PROTOCOL_REL: PROTOCOL_SHA,
    }
    write_json(lock_dir / "component_hash_audit.json", component_audit)

    # Validation comparison tables.
    rows = []
    for d in DOMAINS:
        row = {"dataset": d}
        for prefix, m in (("06D", metrics06d["per_domain"][d]), ("08A", metrics08a["per_domain"][d])):
            for key, val in m.items():
                row[f"{prefix}_{key}"] = val
        rows.append(row)
    pd.DataFrame(rows).to_csv(result_dir / "Table1_06D_vs_08A_validation.csv", index=False)

    # Copy lock into Results for traceability, while Recovery remains self-contained with weights.
    write_json(result_dir / "PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK.json", lock)
    (result_dir / "README.md").write_text(readme, encoding="utf-8")

    result_index = build_index(result_dir)
    lock_index = build_index(lock_dir)

    results_zip = output_root / "PAPEREXP08A_Final_Validation_Refined_Generalized_Model_Results.zip"
    recovery_zip = output_root / "PAPEREXP08A_Final_Locked_Generalized_Model_Recovery.zip"
    zip_tree(result_dir, results_zip)
    zip_tree(lock_dir, recovery_zip)

    print("\n[VALIDATION RESULT]")
    print(json.dumps(metrics08a, indent=2))
    print("\n[LOCKED]", LOCK_ID)
    print("[RESULTS ZIP]", results_zip)
    print("[RESULTS SHA256]", sha_file(results_zip))
    print("[LOCK/RECOVERY ZIP]", recovery_zip)
    print("[LOCK/RECOVERY SHA256]", sha_file(recovery_zip))
    print("[IMPORTANT] No frozen TEST or DERM12345 file was read. This is a validation-selected final configuration lock.")


if __name__ == "__main__":
    main()


[CODE VERSION] PAPEREXP08A_06D_GAIN_1P05_FINAL_CONFIGURATION_LOCK_2026-08-30_v2_KAGGLE_EXTRACTED_RECOVERY_SUPPORT
[LOCK ID] PAPEREXP08A_FINAL_GENERALIZED_CONFIGURATION_LOCK_2026-08-30_v1
[STATUS] DEVELOPMENT VALIDATION REPRODUCTION + FINAL CONFIGURATION LOCK
[BOUNDARY] NO FROZEN TEST OR DERM12345 FILE IS READ BY THIS NOTEBOOK
[06A SOURCE MODE] KAGGLE_EXTRACTED_FILES verified by exact component SHA256
[06D SOURCE MODE] KAGGLE_EXTRACTED_FILES verified by exact component SHA256

[VALIDATION RESULT]
{
  "per_domain": {
    "ISIC2018_HAM10000": {
      "n": 1020,
      "accuracy": 0.8558823529411764,
      "macro_f1": 0.75776524222606,
      "balanced_accuracy": 0.7436588107975358,
      "melanoma_recall": 0.5729166666666666,
      "macro_auc": 0.9606153051514916
    },
    "BCN20000": {
      "n": 2387,
      "accuracy": 0.728110599078341,
      "macro_f1": 0.664624276102102,
      "balanced_accuracy": 0.6552017399367872,
      "melanoma_recall": 0.6462809917355372,
      "macro_auc": 0.90